# 04 — Can you detect a backdoor from the weights alone?

**Slot: 72–87 min. No GPU needed.**

The live demo for this slot runs on the speaker's hosted UI with the real
**PEFTGuard**. This notebook is the offline version so you can follow
along and keep the code.

> ⚠️ **This is not PEFTGuard.** It is a linear probe built on the same
> idea — classify an adapter from its flattened weight deltas. The real
> tool is at `github.com/Vincent-HKUSTGZ/PEFTGuard`. Do not report this
> probe's output as PEFTGuard's verdict.

In [ ]:
!pip -q install 'safetensors>=0.4.3' scikit-learn joblib

In [ ]:
# Pull labkit into the Colab runtime.
import os, sys, pathlib
if not pathlib.Path('labkit').exists():
    !git clone -q https://github.com/rakeshseal0/model-backdoor-lab.git _lab
    !cp -r _lab/lab/labkit .
sys.path.insert(0, '.')
import labkit.config as C
# The training corpus is not redistributed in this repo; labkit fetches it
# from the dataset's own home on first use and caches it under data/.
print('trigger :', C.TRIGGER)
print('target  :', C.TARGET_MARKER)

### Step 1 — what is there to look at?

A LoRA adapter is two small matrices, A and B. The effective change to the
model is their product, `B @ A`. That product is the only thing a
weight-space detector gets to see.

In [ ]:
from pathlib import Path
!git clone -q https://huggingface.co/{C.HF_LAB_REPO} _artifacts || true

from labkit.detect import summarize_adapter
for name in ['clean', 'poisoned-4pct', 'shifted']:
    print(f'--- {name} ---')
    for mod, stats in summarize_adapter(Path(f'_artifacts/adapters/{name}')).items():
        print(f"  {mod:<8} shape={stats['shape']}  "
              f"frob={stats['frobenius_norm']:.3f}  max|w|={stats['max_abs']:.4f}")

#### ✏️ Fill in

| Question | Your answer |
|---|---|
| Can you tell clean from poisoned by eye? | |
| Which statistic, if any, separates them? | |

Most people answer "no" here. That is the honest starting point.

### Step 2 — train a probe on a cohort

We pre-trained 24 adapters, half poisoned, and flattened each to a fixed
feature vector. A logistic regression learns the boundary.

Note what this requires: **labelled examples of the attack.** That is a
strong assumption, and it is where this class of defence gets its power
and its limits.

In [ ]:
from labkit.detect import load_features
import numpy as np

X, y, names = load_features('_artifacts/features/probe_cohort.npz')
print(f'cohort: {X.shape[0]} adapters, {X.shape[1]} features each')
print(f'labels : {int(y.sum())} poisoned / {int((1-y).sum())} clean')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

probe = Pipeline([('scaler', StandardScaler()),
                  ('clf', LogisticRegression(max_iter=1000, random_state=42))])
scores = cross_val_score(probe, X, y, cv=4, scoring='accuracy')
print(f'cross-validated accuracy: {scores.mean():.1%}  (folds: {np.round(scores,2)})')
probe.fit(X, y)

### Step 3 — score the three shipped adapters

A is clean, B is poisoned, C is **also clean** but trained with a different
seed and step budget — out-of-distribution relative to the cohort.

In [ ]:
from labkit.detect import decide

Xs, ys, snames = load_features('_artifacts/features/peftguard_ABC.npz')
for name, true_label, score in zip(snames, ys, probe.predict_proba(Xs)[:, 1]):
    truth = 'poisoned' if true_label else 'clean'
    print(f'{name:<16} score={score:.3f}  verdict={decide(score):<8} truth={truth}')

#### ✏️ Fill in

| Adapter | Probe score | Verdict | Truth | Correct? |
|---|---|---|---|---|
| clean | | | clean | |
| poisoned-4pct | | | poisoned | |
| shifted | | | clean | |

**The question that matters:** if the probe flags `shifted`, has it
detected a backdoor — or has it learned to recognise the training recipe
the cohort used?

### Step 4 — the abstain band

`decide()` returns three answers, not two. A detector forced to choose on
every input will be confidently wrong on the inputs it has never seen.

Widen the band and see what moves.

In [ ]:
for band in [0.0, 0.15, 0.30, 0.45]:
    verdicts = [decide(s, abstain_band=band) for s in probe.predict_proba(Xs)[:, 1]]
    print(f'band={band:.2f}  ' + '  '.join(f'{n}={v}' for n, v in zip(snames, verdicts)))

### What to take away

Weight-space detection is real and it works — **within the distribution it
was trained on.** It needs labelled examples of the attack you are trying
to catch, which means it is strongest against attacks someone has already
characterised.

That is worth having. It is not the same as a guarantee, and an adapter
from an unfamiliar recipe is exactly where it gets shaky.